# DNS Exfiltration Detection — Character-Embedding Autoencoder

**Project:** Multi-agent AI SOC for DNS/SNMP anomaly detection (HIT, *AI in Cybersecurity based on NVIDIA Morpheus*).

This notebook trains and evaluates the DNS detector: a **character-embedding autoencoder** — an unsupervised neural network that learns the lexical "shape" of *benign* domain names and flags names it cannot reconstruct (DNS tunneling / exfiltration).

**Pipeline:** EDA → feature analysis → build & train the model → evaluate (ROC, score margin) → robustness stress-test → export `charae.pt` for the live SOC.

> **Runs on Colab or locally.** It needs one file: `cic_bell_qnames.csv` (per-query qnames + labels parsed from CIC-Bell-DNS-EXF-2021). On Colab: enable a GPU runtime (optional — the model is tiny) and upload that CSV.

In [ ]:
# Colab: torch / sklearn / matplotlib are preinstalled — nothing to pip install.
import os, math, base64, random
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

SEED = 13
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Load the data

Each row is one DNS **query** (not a response): the queried name and its ground-truth label. Benign names are normal lookups; attack names carry exfiltrated data.

In [ ]:
CSV = "cic_bell_qnames.csv"
if not os.path.exists(CSV):                       # running inside the repo instead of Colab
    alt = os.path.join("..", "data", "real", "cic_bell_qnames.csv")
    CSV = alt if os.path.exists(alt) else CSV
df = pd.read_csv(CSV, dtype=str).dropna(subset=["qname"])
df["qname"] = df["qname"].str.lower().str.rstrip(".")
print(len(df), "queries")
print(df["label"].value_counts())
df.head()

## 2. Exploratory Data Analysis (EDA)

Before any model: *is there signal at the per-query level?* We compute classic DNS-tunneling **lexical** features (length, character entropy, digit ratio, vowel ratio…) and compare benign vs attack.

In [ ]:
VOWELS = set("aeiou")

def entropy(s):
    s = s.replace(".", "")
    if not s: return 0.0
    c = Counter(s); n = len(s)
    return -sum((k / n) * math.log2(k / n) for k in c.values())

def lexical(q):
    bare = q.replace(".", ""); labels = [x for x in q.split(".") if x]; n = max(len(bare), 1)
    return {
        "length": len(q),
        "max_label_len": max((len(x) for x in labels), default=0),
        "entropy": entropy(bare),
        "digit_ratio": sum(ch.isdigit() for ch in bare) / n,
        "vowel_ratio": sum(ch in VOWELS for ch in bare) / n,
        "unique_char_ratio": len(set(bare)) / n,
    }

# sample for speed (per class; keeps all columns across pandas versions)
samp = pd.concat([g.sample(min(len(g), 20000), random_state=SEED) for _, g in df.groupby("label")])
feat = pd.DataFrame([lexical(q) for q in samp["qname"]]); feat["label"] = samp["label"].values
print("median feature values by class:")
feat.groupby("label").median()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["length", "entropy", "digit_ratio"]):
    for lab, color in [("benign", "tab:blue"), ("attack", "tab:red")]:
        ax.hist(feat[feat.label == lab][col], bins=40, alpha=0.5, label=lab, color=color, density=True)
    ax.set_title(col); ax.set_xlabel(col); ax.legend()
plt.suptitle("Per-query lexical features: benign (blue) vs attack (red)")
plt.tight_layout(); plt.show()

The distributions barely overlap — benign names are short, vowel-rich, low-entropy; exfil names are long, digit-heavy, high-entropy. **The signal is strong at the per-query level.** (The earlier window-aggregate model scored only 0.75 because averaging over 10s windows blurred this away.)

Now the **correlation heatmap** — which features are redundant vs complementary.

In [ ]:
cols = ["length", "max_label_len", "entropy", "digit_ratio", "vowel_ratio", "unique_char_ratio"]
corr = feat[cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im); plt.title("Lexical feature correlation"); plt.tight_layout(); plt.show()

length, max_label_len and entropy are highly correlated (they all measure "this name is big and random") — a hint that hand-crafted features are redundant. Instead of choosing among them, we let a neural network **learn its own representation** from the raw characters.

## 3. The model: character-embedding autoencoder

- **Embedding layer** — each character → a learned 32-dim vector (the model's own numeric vocabulary).
- **GRU encoder** — reads the name left-to-right into a 16-dim latent "summary".
- **GRU decoder** — rebuilds the name character-by-character from the latent.
- **Loss** = cross-entropy = *how surprised* a benign-trained model is by each character.

Trained on **benign names only** → benign reconstructs cheaply (low score), random exfil is "surprising" (high score). The score *is* the reconstruction error. Includes **dropout** (regularisation), and we train with **minibatches** + **early stopping** (see §4).

In [ ]:
PAD, UNK = 0, 1
MAX_LEN, EMBED, HIDDEN, LATENT = 48, 32, 64, 16

def build_vocab(names):
    chars = sorted({c for n in names for c in n})
    return {c: i + 2 for i, c in enumerate(chars)}     # 0=PAD, 1=UNK

def encode(name, vocab):
    ids = [vocab.get(c, UNK) for c in name[:MAX_LEN]]
    return ids + [PAD] * (MAX_LEN - len(ids))

class CharAE(nn.Module):
    def __init__(self, vocab_size, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMBED, padding_idx=PAD)
        self.enc = nn.GRU(EMBED, HIDDEN, batch_first=True)
        self.to_latent = nn.Linear(HIDDEN, LATENT)
        self.from_latent = nn.Linear(LATENT, HIDDEN)
        self.dec = nn.GRU(EMBED, HIDDEN, batch_first=True)
        self.out = nn.Linear(HIDDEN, vocab_size)
        self.drop = nn.Dropout(dropout)               # regularisation against overfitting

    def forward(self, x, lengths):
        e = self.emb(x)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.enc(packed)
        z = self.to_latent(self.drop(h[-1]))          # bottleneck latent
        h0 = self.from_latent(z).unsqueeze(0)
        dec_in = torch.zeros_like(e); dec_in[:, 1:, :] = e[:, :-1, :]   # teacher forcing
        out, _ = self.dec(dec_in, h0)
        return self.out(out)

In [ ]:
def tensors(names, vocab):
    ids = np.array([encode(n, vocab) for n in names], dtype=np.int64)
    x = torch.tensor(ids, device=device)
    lengths = torch.tensor([max(min(len(n), MAX_LEN), 1) for n in names])
    return x, lengths

def score(model, names, vocab, bs=4096):
    model.eval(); lf = nn.CrossEntropyLoss(ignore_index=PAD, reduction="none"); out = []
    with torch.no_grad():
        for i in range(0, len(names), bs):
            x, lengths = tensors(names[i:i + bs], vocab)
            per = lf(model(x, lengths).transpose(1, 2), x)
            mask = (x != PAD).float()
            out.append((per.sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy())
    return np.concatenate(out)

def train(model, tr, va, vocab, epochs=30, bs=512, lr=1e-3, patience=4):
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    lf = nn.CrossEntropyLoss(ignore_index=PAD)
    x, ln = tensors(tr, vocab); n = len(tr)
    hist = {"train": [], "val": []}; best, best_state, bad = 1e9, None, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n); tot = 0.0
        for i in range(0, n, bs):                      # minibatch SGD
            idx = perm[i:i + bs]
            opt.zero_grad(); loss = lf(model(x[idx], ln[idx]).transpose(1, 2), x[idx])
            loss.backward(); opt.step(); tot += loss.item() * len(idx)
        tr_l = tot / n; va_l = float(score(model, va, vocab).mean())
        hist["train"].append(tr_l); hist["val"].append(va_l)
        print(f"epoch {ep:2d}  train={tr_l:.4f}  val={va_l:.4f}")
        if va_l < best - 1e-4:                          # early stopping
            best, best_state, bad = va_l, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: print("early stop"); break
    if best_state: model.load_state_dict(best_state)
    return hist

## 4. Train

**Split (no leakage):** unique names only; 70% benign → train, 15% benign → validation, 15% benign + **all attack** → test. The model never sees an attack during training (that's what makes it unsupervised / zero-day capable).

In [ ]:
uniq = df.drop_duplicates("qname")
benign = uniq[uniq.label == "benign"]["qname"].tolist()
attack = uniq[uniq.label == "attack"]["qname"].tolist()
random.shuffle(benign)
n_tr, n_va = int(0.7 * len(benign)), int(0.15 * len(benign))
tr, va, te_b = benign[:n_tr], benign[n_tr:n_tr + n_va], benign[n_tr + n_va:]
print(f"train={len(tr)}  val={len(va)}  test_benign={len(te_b)}  attack(test)={len(attack)}")

vocab = build_vocab(tr)
model = CharAE(len(vocab) + 2)
print("trainable params:", sum(p.numel() for p in model.parameters()))
hist = train(model, tr, va, vocab)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist["train"], label="train", marker="o")
plt.plot(hist["val"], label="validation", marker="s")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")
plt.title("Training vs validation loss")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Reading this curve (over/underfitting):** train and validation fall *together* → healthy. If validation had turned *upward* while train kept dropping, that's **overfitting** (memorising). If both stalled high, that's **underfitting** (too weak / under-trained). Dropout, early stopping and minibatching are what keep us in the healthy regime.

## 5. Evaluation

The model outputs an anomaly **score** per name. The **ROC curve** sweeps every possible threshold and plots recall (caught attacks) vs false-positive rate (wrongly-flagged benign). **ROC-AUC** = area under it = probability a random attack scores above a random benign.

In [ ]:
test_names = te_b + attack
y = np.array([0] * len(te_b) + [1] * len(attack))
s = score(model, test_names, vocab)
auc = roc_auc_score(y, s); ap = average_precision_score(y, s)
fpr, tpr, _ = roc_curve(y, s)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"char-AE (AUC = {auc:.3f})", color="tab:green")
plt.plot([0, 1], [0, 1], "--", color="gray", label="random (0.5)")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC curve — DNS exfil detection"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"ROC-AUC = {auc:.3f}   PR-AUC = {ap:.3f}")

In [ ]:
def recall_at_fpr(y, s, fpr=0.01):
    b = np.sort(s[y == 0]); thr = b[int((1 - fpr) * len(b)) - 1]
    return float((s[y == 1] > thr).mean()), thr

r1, thr1 = recall_at_fpr(y, s, 0.01)
r01, _ = recall_at_fpr(y, s, 0.001)
print(f"recall@1%FPR = {r1:.3f}    recall@0.1%FPR = {r01:.3f}")

plt.figure(figsize=(8, 4))
plt.hist(s[y == 0], bins=60, alpha=0.6, label="benign", color="tab:blue", density=True)
plt.hist(s[y == 1], bins=60, alpha=0.6, label="attack", color="tab:red", density=True)
plt.axvline(thr1, color="black", ls="--", label="1% FPR threshold")
plt.xlabel("anomaly score (reconstruction cross-entropy)"); plt.ylabel("density")
plt.title("Score distributions: benign vs attack (the separation margin)")
plt.legend(); plt.show()

A near-perfect AUC on this data is **honest but unflattering** — CIC-Bell's exfil is lexically extreme. A serious evaluation asks: *how does recall hold up as the attacker gets stealthier?*

## 6. Robustness stress-test

We synthesise progressively stealthier exfil and measure recall at the model's fixed **1%-FPR threshold**. The stealthiest family encodes data as **real benign words mined from this very dataset** — a fair, hard attack.

In [ ]:
def b32(nb): return base64.b32encode(np.random.randint(0,256,nb,dtype=np.uint8).tobytes()).decode().lower().rstrip("=")
def hx(nb):  return np.random.randint(0,256,nb,dtype=np.uint8).tobytes().hex()

domains, words = [], Counter()
for q in benign:
    labels = [x for x in q.split(".") if x]
    if 2 <= len(labels) <= 3 and all(x.replace("-", "").isalnum() for x in labels):
        domains.append(".".join(labels[-2:]))
    for x in labels:
        if x.isalpha() and 3 <= len(x) <= 10: words[x] += 1
domains = list(dict.fromkeys(domains)) or ["example.com"]
wl = [w for w, _ in words.most_common(512)] or ["host", "mail"]
def dom(): return domains[np.random.randint(len(domains))]
def wjoin(k): return "".join(wl[i] for i in np.random.randint(0, len(wl), k))

NS = 3000
families = {
    "overt_base32_40": lambda: f"{b32(25)[:40]}.{dom()}",
    "hex_40":          lambda: f"{hx(20)[:40]}.{dom()}",
    "b32_chunked_8":   lambda: ".".join(b32(25)[i:i+8] for i in range(0,32,8)) + f".{dom()}",
    "short_b32_8":     lambda: f"{b32(5)[:8]}.{dom()}",
    "word4_label":     lambda: f"{wjoin(4)}.{dom()}",
    "word2_label":     lambda: f"{wjoin(2)}.{dom()}",
}
stress = {fam: float((score(model, [g() for _ in range(NS)], vocab) > thr1).mean())
          for fam, g in families.items()}
stress

In [ ]:
order = list(families.keys())
plt.figure(figsize=(9, 4))
plt.bar(order, [stress[f] for f in order], color="tab:purple")
plt.ylabel("recall @ 1% FPR"); plt.ylim(0, 1.08)
plt.title("Robustness: recall vs exfil stealthiness  (overt → stealthy)")
plt.xticks(rotation=30, ha="right")
for i, f in enumerate(order):
    plt.text(i, stress[f] + 0.02, f"{stress[f]:.2f}", ha="center")
plt.tight_layout(); plt.show()

**The honest envelope:** ~100% recall on high-entropy exfil (overt / hex / chunked), degrading sharply on **word-encoded** names. This is a *fundamental* lexical limit — word-encoded exfil is, by construction, lexically benign. The defence is the complementary **behavioural** detector (query volume/rate), since word-encoding still needs many queries to move data. Lexical + behavioural = defence in depth.

## 7. Export for the live SOC

Save weights + vocabulary + the chosen FPR thresholds into one file. The SOC loads this for inference — **no retraining in production**.

In [ ]:
ckpt = {
    "state_dict": model.state_dict(),
    "vocab": vocab,
    "thr_1pct": float(thr1),
    "thr_01pct": float(np.sort(s[y == 0])[int(0.999 * (y == 0).sum()) - 1]),
}
torch.save(ckpt, "charae.pt")
print("saved charae.pt")
# On Colab, download it:
# from google.colab import files; files.download("charae.pt")
# Then drop charae.pt into the project's  models/  folder.

## Summary

- **Representation beats algorithm:** raw characters + learned embeddings >> hand-crafted window aggregates (0.75 → ~1.0 ROC-AUC).
- **Unsupervised autoencoder** trained on benign only → flags zero-day tunneling without ever seeing an attack.
- **Trained properly:** train/val/test split, dropout, minibatch SGD, early stopping → healthy loss curves, no over/underfitting.
- **Evaluated honestly:** not a single rosy number, but a recall-vs-stealthiness envelope that exposes the real limit (word-encoded exfil) — motivating the complementary behavioural detector.
- **Note — SNMP** uses a separate *behavioural* autoencoder (no rich string to read), trained on synthetic traffic (no public real SNMP-attack dataset exists).